# Notebook 01 — Exploratory Data Analysis: AfriSenti Amharic

**Project**: Amharic Sentiment Analysis Platform  
**Author**: Eyob Nebyou  
**Dataset**: AfriSenti-SemEval 2023 — Amharic Subset  
**Goal**: Understand the dataset before any modeling.

---

## What We're Answering

1. What does the dataset look like? (size, splits, columns)
2. How balanced are the sentiment classes?
3. What does Amharic text look like in this dataset?
4. How long are the tweets? (text length distribution)
5. What are the most common words per sentiment class?
6. What challenges does Amharic NLP present?

---

In [ ]:
# ── Imports ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from collections import Counter

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)
print('Libraries loaded successfully.')

## 1. Load the Dataset

In [ ]:
# ── Load AfriSenti Amharic from HuggingFace ───────────────────
print('Loading AfriSenti Amharic dataset...')
dataset = load_dataset(
    'shmuhammad/AfriSenti-twitter-sentiment',
    'amh',
    trust_remote_code=True
)
print(dataset)

In [ ]:
# ── Convert to pandas DataFrames ──────────────────────────────
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f'Train set:      {train_df.shape}')
print(f'Validation set: {val_df.shape}')
print(f'Test set:       {test_df.shape}')
print(f'Total samples:  {len(train_df) + len(val_df) + len(test_df):,}')
print(f'\nColumns: {train_df.columns.tolist()}')
print(f'\nSample rows:')
print(train_df.head())

In [ ]:
# ── Label mapping ──────────────────────────────────────────────
# Check what the labels look like
print('Unique labels in train:', train_df['label'].unique())
print('Label dtype:', train_df['label'].dtype)

# Map numeric labels to text if needed
label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
if train_df['label'].dtype in ['int64', 'int32']:
    train_df['sentiment'] = train_df['label'].map(label_map)
    val_df['sentiment'] = val_df['label'].map(label_map)
    test_df['sentiment'] = test_df['label'].map(label_map)
else:
    train_df['sentiment'] = train_df['label']
    val_df['sentiment'] = val_df['label']
    test_df['sentiment'] = test_df['label']

print('\nSentiment value counts (train):')
print(train_df['sentiment'].value_counts())

## 2. Class Distribution — Is the Dataset Balanced?

In [ ]:
# ── Class distribution across all splits ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'positive': '#1D9E75', 'negative': '#E24B4A', 'neutral': '#534AB7'}

for ax, (split_name, split_df) in zip(axes, [
    ('Train', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]):
    counts = split_df['sentiment'].value_counts()
    bar_colors = [colors.get(l, '#888') for l in counts.index]
    bars = ax.bar(counts.index, counts.values, color=bar_colors, alpha=0.85)
    ax.set_title(f'{split_name} ({len(split_df):,} samples)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        pct = val / len(split_df) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{val}\n({pct:.1f}%)', ha='center', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Sentiment Class Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_class_distribution.png')

**Finding**: Write your observation here after running the cell.  
Example: *"The dataset is imbalanced with X% positive, Y% negative, Z% neutral. This will affect our choice of evaluation metric — we will use macro F1 which treats all classes equally."*

## 3. Sample Amharic Texts

Let's look at actual Amharic tweets to understand what we're working with.

In [ ]:
# ── Show sample texts per class ────────────────────────────────
print('Sample texts from each sentiment class:\n')
for sentiment in ['positive', 'negative', 'neutral']:
    print(f'=== {sentiment.upper()} ===')
    samples = train_df[train_df['sentiment'] == sentiment]['text'].head(3)
    for i, text in enumerate(samples, 1):
        print(f'  {i}. {text}')
    print()

## 4. Text Length Analysis

In [ ]:
# ── Character and word length distributions ───────────────────
train_df['char_length'] = train_df['text'].str.len()
train_df['word_count'] = train_df['text'].str.split().str.len()

print('Text length statistics (train):')
print(train_df[['char_length', 'word_count']].describe().round(2))

In [ ]:
# ── Length distribution by sentiment ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_list = ['#1D9E75', '#E24B4A', '#534AB7']
sentiments = ['positive', 'negative', 'neutral']

for ax, metric, title in zip(
    axes,
    ['char_length', 'word_count'],
    ['Character Length', 'Word Count']
):
    for sentiment, color in zip(sentiments, colors_list):
        subset = train_df[train_df['sentiment'] == sentiment][metric]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=sentiment, density=True)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.set_title(f'{title} by Sentiment', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/01_text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_text_length_distribution.png')

## 5. Most Common Words Per Sentiment

In [ ]:
# ── Top words per sentiment class ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, sentiment, color in zip(axes, sentiments, colors_list):
    texts = train_df[train_df['sentiment'] == sentiment]['text']
    all_words = ' '.join(texts).split()
    # Filter out very short tokens
    all_words = [w for w in all_words if len(w) > 1]
    word_counts = Counter(all_words).most_common(15)
    words, counts = zip(*word_counts)

    ax.barh(range(len(words)), counts, color=color, alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    ax.set_title(f'Top 15 Words — {sentiment.capitalize()}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/01_top_words_per_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_top_words_per_sentiment.png')

## 6. Amharic NLP Challenges

Understanding what makes Amharic hard for NLP is important context for our modeling choices.

In [ ]:
# ── Check for mixed scripts (Amharic + Latin/English) ─────────
import re

def has_amharic(text):
    return bool(re.search(r'[\u1200-\u137F]', str(text)))

def has_latin(text):
    return bool(re.search(r'[a-zA-Z]', str(text)))

train_df['has_amharic'] = train_df['text'].apply(has_amharic)
train_df['has_latin'] = train_df['text'].apply(has_latin)
train_df['is_mixed'] = train_df['has_amharic'] & train_df['has_latin']

print('Script analysis (train set):')
print(f'  Pure Amharic (Ethiopic only): {train_df["has_amharic"] & ~train_df["has_latin"].sum():,} ({(train_df["has_amharic"] & ~train_df["has_latin"]).mean():.1%})')
print(f'  Mixed (Amharic + Latin):      {train_df["is_mixed"].sum():,} ({train_df["is_mixed"].mean():.1%})')
print(f'  Latin only:                   {(~train_df["has_amharic"] & train_df["has_latin"]).sum():,}')

print('\nSample mixed-script tweets:')
mixed_samples = train_df[train_df['is_mixed']]['text'].head(3)
for i, text in enumerate(mixed_samples, 1):
    print(f'  {i}. {text}')

In [ ]:
# ── Check for URLs, mentions, hashtags ────────────────────────
train_df['has_url'] = train_df['text'].str.contains(r'http\S+', regex=True)
train_df['has_mention'] = train_df['text'].str.contains(r'@\w+', regex=True)
train_df['has_hashtag'] = train_df['text'].str.contains(r'#\w+', regex=True)

print('Noise analysis (train set):')
print(f'  Tweets with URLs:     {train_df["has_url"].sum():,} ({train_df["has_url"].mean():.1%})')
print(f'  Tweets with mentions: {train_df["has_mention"].sum():,} ({train_df["has_mention"].mean():.1%})')
print(f'  Tweets with hashtags: {train_df["has_hashtag"].sum():,} ({train_df["has_hashtag"].mean():.1%})')

## 7. Save Processed DataFrames

In [ ]:
# ── Save all splits for use in next notebooks ──────────────────
import os
os.makedirs('../data/processed', exist_ok=True)

train_df.to_parquet('../data/processed/train_raw.parquet', index=False)
val_df.to_parquet('../data/processed/val_raw.parquet', index=False)
test_df.to_parquet('../data/processed/test_raw.parquet', index=False)

print('Saved:')
print(f'  train_raw.parquet — {train_df.shape}')
print(f'  val_raw.parquet   — {val_df.shape}')
print(f'  test_raw.parquet  — {test_df.shape}')
print('\n✅ EDA complete. Proceed to Notebook 02 for preprocessing.')

## 8. Summary & Findings

Fill this in after running all cells:

| Item | Detail |
|---|---|
| Total samples | Fill in |
| Train / Val / Test | Fill in |
| Class balance | Fill in |
| Most common sentiment | Fill in |
| Avg tweet length (words) | Fill in |
| Mixed script tweets | Fill in |
| Tweets with URLs | Fill in |

**Key NLP challenges identified**:
- Mixed Amharic/Latin script in many tweets
- URLs, mentions, hashtags need cleaning
- Class imbalance may require weighted loss or oversampling
- Amharic morphology is complex — one root word has many forms

**Next steps (Day 3 — Preprocessing)**:
- Remove URLs, mentions, hashtags
- Normalize Ethiopic characters
- Handle mixed script text
- Save clean dataset ready for modeling